# Imports & paths

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

from lifetimes import BetaGeoFitter

DATA_INTERIM = Path("../data/interim")
DATA_PROCESSED = Path("../data/processed")
ART_MODELS = Path("../artifacts/models")
ART_SCORES = Path("../artifacts/scores")

# Load data

In [12]:
df_txn = pd.read_parquet(DATA_INTERIM / "transactions_clean.parquet")
labels = pd.read_parquet(DATA_PROCESSED / "churn_prediction_labels.parquet")

# Tính x, t_x, T cho BG-NBD

In [13]:
from lifetimes.utils import summary_data_from_transaction_data

summary = summary_data_from_transaction_data(
    df_txn,
    customer_id_col="customer_id",
    datetime_col="transaction_date",
    monetary_value_col="amount",
    observation_period_end="2025-12-31"
)
summary = summary.reset_index() 

In [14]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(
    summary["frequency"],
    summary["recency"],
    summary["T"]
)

summary["p_alive"] = bgf.conditional_probability_alive(
    summary["frequency"],
    summary["recency"],
    summary["T"]
)

In [15]:
summary["exp_txn_90d"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    90,
    summary["frequency"],
    summary["recency"],
    summary["T"]
)



In [16]:
summary.head()

,customer_id,frequency,recency,T,monetary_value,p_alive,exp_txn_90d
0,C00000,11.0,112.0,112.0,93.370000,0.970418,7.934405
1,C00001,17.0,278.0,289.0,68.373529,0.958491,5.037573
2,C00002,9.0,37.0,133.0,86.307778,0.000266,0.001586
3,C00003,3.0,45.0,88.0,19.393333,0.538345,1.962628
4,C00004,17.0,98.0,206.0,109.965882,0.000132,0.000945


# Evaluation

In [17]:
eval_df = summary.merge(labels[["customer_id", "churn_90d"]], on="customer_id", how="inner")
eval_df.groupby("churn_90d")["p_alive"].mean()

churn_90d
False    0.702411
True     0.128930
Name: p_alive, dtype: float64

- p-alive của churn user thấp hơn rõ rệt so với non-churn 

In [18]:
eval_df.groupby("churn_90d")[["p_alive", "exp_txn_90d"]].mean()

,p_alive,exp_txn_90d
churn_90d,,
False,0.702411,7.174452
True,0.128930,0.126519


# Log

In [19]:

bgf.save_model(ART_MODELS / "bgnbd_model.pkl")

out = summary[[
    "customer_id",
    "p_alive",
    "exp_txn_90d"
]].copy()

out.to_parquet(
    ART_SCORES / "bgnbd_90d.parquet",
    index=False
)